In [1]:
!pip install transformers
!pip install accelerate
!pip install qwen-vl-utils
!pip install bitsandbytes
!pip install safetensors


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.2 MB/s eta 0:00:00


In [2]:
!pip install faster-whisper


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.6 MB/s eta 0:00:00


In [3]:
!pip install edge-tts
!pip install nest_asyncio


In [4]:
!pip install gradio


In [1]:
!pip install llama-index
!pip install llama-index-embeddings-huggingface
!pip install sentence-transformers


In [2]:
import os

base_dir = "/content/tanit-multimodal-fertility-assistant"

os.makedirs(f"{base_dir}/voice", exist_ok=True)
os.makedirs(f"{base_dir}/rag", exist_ok=True)
os.makedirs(f"{base_dir}/data", exist_ok=True)

print("Structure créée :")
!tree /content/tanit-multimodal-fertility-assistant


Structure créée :
/bin/bash: line 1: tree: command not found


In [3]:
%%writefile /content/tanit-multimodal-fertility-assistant/voice/stt.py
import os
import torch
from faster_whisper import WhisperModel #moteur STT basé sur whisper
from pydub import AudioSegment #pour charger l'audio

class STTEngine:
    def __init__(self, model_size="medium", device="cuda"):


        print(f" STT : Chargement de Whisper ({model_size}) sur GPU...")
        try:
            #chargement du modele whisper sur GPU en float16
            self.model = WhisperModel(model_size, device="cuda", compute_type="float16")
            print(" Whisper GPU chargé.")
        except Exception as e:
            print(f" GPU indisponible, fallback CPU : {e}")
            self.model = WhisperModel(model_size, device="cpu", compute_type="int8")

    def transcribe(self, audio_path):
        #une chaine vide renvoyeé si aucun fichier audio n'est fourni
        if not audio_path: return ""

        #fichier temporaire netoyee
        clean_path = "/content/temp_clean.wav"
        try:
            #chargement et normalisation de l'audio
            audio = AudioSegment.from_file(audio_path)
            audio = audio.set_frame_rate(16000).set_channels(1)
            audio.export(clean_path, format="wav")
        except:
            clean_path = audio_path

        try:
            #transcription avec beam search et sans VAD
            #beam search pour generer le text , les 5 meilleurs hypotheses de transcription (beam_size=5)
            segments, _ = self.model.transcribe(clean_path, beam_size=5, vad_filter=False)
            #concatenation du text
            text = " ".join([s.text for s in segments]).strip()
            return text
        except Exception as e:
            print(f" Erreur STT : {e}")
            return ""


Writing /content/tanit-multimodal-fertility-assistant/voice/stt.py


In [4]:
%%writefile /content/tanit-multimodal-fertility-assistant/voice/tts.py
import edge_tts #bibliotheque Microsoft Edge TTS pour generer l'audio a partir du text
import asyncio
import nest_asyncio
import tempfile
import os


nest_asyncio.apply()

#Dictionnaire des voix disponnibles par langue
VOICES = {
    "fr": "fr-FR-HenriNeural", #voix francaise
    "ar": "ar-SA-ZariyahNeural" #voix arabe
}
#fonction asynchrone pour generer l'audio
async def generate_audio(text: str, language: str = "ar") -> str | None:
    """
    Génère un fichier audio MP3 à partir d'un texte.

    Args:
        text (str): texte à convertir en audio.
        language (str): "ar" pour arabe, "fr" pour français.

    Returns:
        str | None: chemin vers le fichier audio généré ou None si échec.
    """

    if not text or not text.strip():
        print(" Texte vide fourni au TTS.")
        return None

    #choix de la voix a partir de la langue demandee et par defaut arabe
    voice = VOICES.get(language.lower(), VOICES["ar"])

    #fichier temporaire pour stocker l'audio genereé
    output_path = tempfile.mktemp(suffix=".mp3")

    try:
        #genere de l'audio avec edge_tts et l'enregistre
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(output_path)


        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            return output_path
        else:
            print(" Aucun audio généré. Vérifiez les paramètres de voix.")
            return None
    except Exception as e:
        print(f" Erreur TTS : {e}")
        return None
#fonction sysnchrone pour rappeler la fonction asynchrone
def speak(text: str, language: str = "ar") -> str | None:
    """
    Fonction synchrone pour générer l'audio.

    Args:
        text (str): texte à convertir.
        language (str): "ar" ou "fr".

    Returns:
        str | None: chemin du fichier MP3 généré
    """
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

    return loop.run_until_complete(generate_audio(text, language))


Writing /content/tanit-multimodal-fertility-assistant/voice/tts.py


In [5]:
%%writefile /content/tanit-multimodal-fertility-assistant/rag/build_index.py
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings #pour creer un index vectoriel pour la recherche semantique
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter

PDF_DIR = "/content/tanit-multimodal-fertility-assistant/data"
#ou sauvegarder l'index vectoriel
INDEX_DIR = "/content/tanit-multimodal-fertility-assistant/rag/graphrag_index"

def build_knowledge_base():
    print(" Initialisation embedding...")
    # modele d'embeddings utilisé par llama_index
    Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-base")
    Settings.llm = None
    # Verification de l'existance de pdfs
    if not os.path.exists(PDF_DIR) or not os.listdir(PDF_DIR):
        print(f" Aucun PDF trouvé : {PDF_DIR}")
        return
    # chargement des pdfs
    documents = SimpleDirectoryReader(PDF_DIR).load_data()
    print(f" {len(documents)} pages chargées.")
    #configuration de decoupages des chunks
    parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)
    #creation de l'index vectoriel
    index = VectorStoreIndex.from_documents(documents, transformations=[parser])
    #sauvegarde de l'index vectoriel
    index.storage_context.persist(persist_dir=INDEX_DIR)
    print(f" Index créé dans {INDEX_DIR}")

if __name__ == "__main__":
    build_knowledge_base()


Writing /content/tanit-multimodal-fertility-assistant/rag/build_index.py


In [6]:
%%writefile /content/tanit-multimodal-fertility-assistant/app.py
import os
import torch
import nest_asyncio
import gradio as gr
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from voice.stt import STTEngine
from voice.tts import speak

nest_asyncio.apply()

# 1. Configuration du device et chemin RAG
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RAG_DIR = "rag/graphrag_index"
print(f" Tanit (Fertility Expert) démarre sur : {DEVICE}")

# 2.Chargement du modèle Qwen2-VL 7B avec quantization 4-bit
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f" Chargement de {MODEL_ID}...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=bnb_config, torch_dtype=torch.float16
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

# 3. Initialisation STT et embeddings
stt = STTEngine(device="cuda")
Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-base")# Modèle d'embeddings multilingue pour transformer le texte en vecteurs sémantiques
# 4. Chargement de l’index RAG
HAS_RAG = False
rag_retriever = None
if os.path.exists(RAG_DIR):
    try:
        rag_index = load_index_from_storage(StorageContext.from_defaults(persist_dir=RAG_DIR))
        rag_retriever = rag_index.as_retriever(similarity_top_k=2) #on prend les 2 documents les plus proches
        HAS_RAG = True
        print(" RAG chargé.")
    except:
        print(" RAG non trouvé ou erreur.")


# 5. Prompt système pour eliminer l'allucination et voir une bonne reponse

BASE_SYSTEM_PROMPT = """Tu es Tanit, une assistante médicale experte en fertilité et gynécologie.

IDENTITÉ ET RÈGLES DE SÉCURITÉ :
1. Tu n'es pas humaine, tu es une IA d'analyse.
2. CRITIQUE : NE JAMAIS INVENTER DE CHIFFRES. Si une valeur est floue, manquante ou illisible, dis explicitement "Valeur non visible" ou "Illisible".
3. Ne fais jamais de diagnostic vital (cancer, urgence vitale). Renvoie vers un médecin.

MODE OPÉRATOIRE :

CAS A : ANALYSE D'IMAGE (Bilan Hormonal, Spermogramme, Écho)
- Lis le document ligne par ligne.
- Repère la colonne "Résultat" et la colonne "Valeur de référence" (Norme).
- Compare le résultat à la norme fournie SUR LE PAPIER (pas tes connaissances générales, car les labos diffèrent).
- Signale uniquement ce qui est ANORMAL (Haut ou Bas).
- Si tout est normal, dis : "Les résultats visibles semblent dans les normes du laboratoire."

CAS B : QUESTION GÉNÉRALE (Sans image)
- Réponds de manière pédagogique, courte et empathique.
- Base-toi sur les connaissances médicales standards en fertilité.
"""
#detecte si le text contient arabe , pour choisir la langue de reponse
def contains_arabic(text):
    if not text: return False
    return any("\u0600" <= c <= "\u06FF" for c in text)

# 6. fonction chat
def chat(audio, image, history):
    if history is None: history = []

    logs = ""
    user_text = ""

    # Transcription Audio
    if audio:
        try:
            user_text = stt.transcribe(audio)
            logs += f"[Audio: {user_text}] "
        except Exception as e:
            logs += f"[Erreur Audio: {e}] "


    if not user_text and not image:
        return history, None, "Veuillez parler ou envoyer une image."

    # detection de la langue
    is_arabic_query = contains_arabic(user_text)

    # Injection dynamique de la langue et du ton
    if is_arabic_query:
        lang_instruction = """
        \nIMPORTANT : L'utilisateur parle ARABE.
        1. Tu DOIS répondre en ARABE (clair et empathique).
        2. Traduis les termes médicaux techniques mais explique-les simplement en arabe.
        3. Ne réponds PAS en français.
        """
    else:
        lang_instruction = """
        \nIMPORTANT : Réponds en FRANÇAIS.
        Utilise un ton professionnel mais rassurant.
        """

    final_system_prompt = BASE_SYSTEM_PROMPT + lang_instruction

    #  Récupération contexte RAG
    context_str = ""
    if HAS_RAG and user_text:
        try:
            nodes = rag_retriever.retrieve(user_text)
            context_str = "\n".join([n.get_content() for n in nodes])
            logs += "[RAG utilisé] "
        except:
            pass

    if context_str:
        final_system_prompt += f"\n\nCONTEXTE MÉDICAL VÉRIFIÉ :\n{context_str}"

    #  Construction des messages pour Qwen
    content_user = []
    if image:
        content_user.append({"type": "image", "image": image})
        if not user_text:
            if is_arabic_query:
                prompt_text = "قم بتحليل هذا التقرير الطبي بدقة. اذكر فقط القيم غير الطبيعية. إذا كانت الصورة غير واضحة، قل ذلك."
            else:
                prompt_text = "Analyse ce rapport médical strictement. Cite les valeurs, compare-les aux références de l'image. Dis-moi ce qui est anormal. Si illisible, dis 'Illisible'."
        else:
            prompt_text = user_text
        content_user.append({"type": "text", "text": prompt_text})
    else:
        content_user.append({"type": "text", "text": user_text})

    messages = [
        {"role": "system", "content": final_system_prompt},
        {"role": "user", "content": content_user}
    ]

    # Génération de réponse
    try:
        text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text_input],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(DEVICE)

        generated = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.01,
            do_sample=True,
            top_p=0.8
        )
        response = processor.batch_decode(generated, skip_special_tokens=True)[0].split("assistant")[-1].strip()

    except Exception as e:
        response = "Une erreur technique empêche l'analyse. Veuillez réessayer avec une image plus claire."
        logs += f"[Erreur Gen: {e}]"

    #  Génère l’audio du texte de réponse selon la langue détectée
    tts_lang = "ar" if contains_arabic(response) else "fr"
    audio_out = speak(response, tts_lang)

    display_text = user_text if user_text else "[Analyse Document]"
    history.append([display_text, response])
    return history, audio_out, logs


# Interface Gradio

custom_css = """
.chatbot {
    border-radius: 20px !important;
    box-shadow: 0 4px 20px rgba(233, 30, 99, 0.1) !important;
    border: 1px solid #fce4ec !important;
}
.gradio-container {
    max-width: 1000px !important;
    margin: 0 auto;
    background-color: #fafafa;
}
h1 {
    background: linear-gradient(45deg, #e91e63, #ff80ab);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
footer {display: none !important;}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="pink",
        secondary_hue="rose",
        font=gr.themes.GoogleFont("Quicksand")
    ),
    css=custom_css,
    title="Tanit - Assistant Fertilité"
) as demo:

    gr.HTML("""
    <div style="text-align: center; padding: 25px 0;">
        <h1 style="font-size: 3rem; margin-bottom: 10px;">🌸 Tanit</h1>
        <p style="color: #666; font-size: 1.1rem; font-weight: 500;">
            Assistant d'Analyse Fertilité & Gynécologie
        </p>
        <div style="display: flex; justify-content: center; gap: 10px; margin-top: 10px;">
            <span style="background: #fce4ec; color: #e91e63; padding: 4px 12px; border-radius: 15px; font-size: 0.8rem;">Bilingue Arabe/Français</span>
            <span style="background: #e3f2fd; color: #1976d2; padding: 4px 12px; border-radius: 15px; font-size: 0.8rem;">Analyse Vision IA</span>
        </div>
    </div>
    """)


    chatbot = gr.Chatbot(
        height=550,
        avatar_images=(
            None,
            "https://cdn-icons-png.flaticon.com/512/3304/3304567.png"
        ),
        show_copy_button=True,
        show_share_button=False,
        bubble_full_width=False,
        render_markdown=True,
        elem_classes="chatbot"
    )

    with gr.Row(variant="panel"):
        with gr.Column(scale=4):
            with gr.Tabs():
                with gr.TabItem("🎤 Audio & Image"):
                    audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Message Vocal")
                    image_in = gr.Image(type="filepath", label="Document Médical", height=250)

            gr.Examples(
                examples=[
                    ["J'ai 35 ans, mon AMH est à 0.5, est-ce inquiétant ?", None],
                    ["هل نتيجة تحليل FSH 12 تعتبر طبيعية؟", None],
                    ["Analyse ce cycle menstruel s'il te plait.", None]
                ],
                inputs=[audio_in, image_in],
                label="Questions types :"
            )

            with gr.Row():
                clear_btn = gr.Button("🗑️ Effacer", variant="secondary", size="lg")
                btn = gr.Button("🌸 Envoyer / إرسال", variant="primary", size="lg")

        with gr.Column(scale=3):
            gr.Markdown("### 🔊 Réponse Vocale")
            audio_out = gr.Audio(label=None, autoplay=True, interactive=False)

            with gr.Accordion("⚙️ Détails Techniques", open=False):
                debug_box = gr.Textbox(label="Logs", interactive=False, lines=6)

    gr.HTML("""
    <div style="text-align: center; margin-top: 20px; color: #999; font-size: 0.8rem; border-top: 1px solid #eee; padding-top: 10px;">
        ⚠️ <strong>Avis de non-responsabilité :</strong> Tanit est une IA d'assistance. Elle ne remplace pas un médecin.
    </div>
    """)

    btn.click(chat, inputs=[audio_in, image_in, chatbot], outputs=[chatbot, audio_out, debug_box])
    clear_btn.click(lambda: ([], None, None, ""), outputs=[chatbot, audio_out, image_in, debug_box])

if __name__ == "__main__":
    demo.launch(share=True)


Writing /content/tanit-multimodal-fertility-assistant/app.py


In [7]:
%cd /content/tanit-multimodal-fertility-assistant
!python rag/build_index.py


/content/tanit-multimodal-fertility-assistant
2025-12-06 14:06:02.979959: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765029962.999756    2773 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765029963.006438    2773 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765029963.022672    2773 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765029963.022695    2773 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765029963.022701    2773 com

In [8]:
%cd /content/tanit-multimodal-fertility-assistant
!python app.py



/content/tanit-multimodal-fertility-assistant
2025-12-06 14:08:09.558872: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765030089.598642    3367 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765030089.608809    3367 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765030089.633819    3367 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765030089.633847    3367 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765030089.633853    3367 com